In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")

PyTorch version: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -q transformers datasets sentencepiece accelerate nltk

In [ ]:
from datasets import load_dataset

# Load the Parquet-compatible C4_200M dataset
ds = load_dataset(
    "martinsr/c4_200m",
    split="train",
    streaming=True
)

print("Dataset loaded successfully!")

README.md:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/62 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/62 [00:00<?, ?it/s]

Dataset loaded successfully!


In [ ]:
sample_data = list(ds.take(5))

for i, item in enumerate(sample_data):
    print(f"\nExample {i+1}")
    print("Incorrect:", item["input"])
    print("Correct  :", item["output"])
    print("-" * 50)


Example 1
Incorrect: Bitcoin is for $7,094 this morning, which CoinDesk says.
Correct  : Bitcoin goes for $7,094 this morning, according to CoinDesk.
--------------------------------------------------

Example 2
Incorrect: The effect of widespread dud targets two face up attack position monsters on the field.
Correct  : 1. The effect of "widespread dud" targets two face up attack position monsters on the field.
--------------------------------------------------

Example 3
Incorrect: tax on sales of stores for non residents are set at 21% for 2014 and 20% in 2015 payable on sales tentatively earned from the difference of the property value some time of purchase (price differences according to working time) and theyear to which sale couples (sales costs), based on the approved annual on the base approved by law).
Correct  : Capital Gains tax on the sale of properties for non-residents is set at 21% for 2014 and 20% in 2015 payable on profits earned on the difference of the property valu

In [ ]:
# Collect 5,000 sentence pairs
pairs = [
    (item["input"], item["output"])
    for item in ds.take(5000)
]

# Remove empty or invalid entries
pairs = [
    (incorrect, correct)
    for incorrect, correct in pairs
    if isinstance(incorrect, str)
    and isinstance(correct, str)
    and incorrect.strip()
    and correct.strip()
]

print("Total training pairs:", len(pairs))

print("\nSample training pair:")
print("Incorrect:", pairs[0][0])
print("Correct  :", pairs[0][1])

Total training pairs: 5000

Sample training pair:
Incorrect: Bitcoin is for $7,094 this morning, which CoinDesk says.
Correct  : Bitcoin goes for $7,094 this morning, according to CoinDesk.


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Load tokenizer
model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load pre-trained T5 model
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Move model to GPU
model = model.to("cuda")

print("T5-small model loaded successfully!")
print("Model device:", next(model.parameters()).device)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5-small model loaded successfully!
Model device: cuda:0


In [ ]:
from datasets import Dataset

# Convert our sentence pairs into a Hugging Face Dataset
train_dataset = Dataset.from_dict({
    "input": [item[0] for item in pairs],
    "output": [item[1] for item in pairs]
})

# Prepare input and output tokens
def preprocess_function(examples):

    inputs = [
        "grammar: " + sentence
        for sentence in examples["input"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=64,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["output"],
        max_length=64,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


# Apply tokenization
tokenized_dataset = train_dataset.map(
    preprocess_function,
    batched=True
)

print("Tokenization completed!")
print("Total examples:", len(tokenized_dataset))
print("\nTokenized sample:")
print(tokenized_dataset[0])

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenization completed!
Total examples: 5000

Tokenized sample:
{'input': 'Bitcoin is for $7,094 this morning, which CoinDesk says.', 'output': 'Bitcoin goes for $7,094 this morning, according to CoinDesk.', 'input_ids': [19519, 10, 9310, 19, 21, 11301, 6, 4198, 591, 48, 1379, 6, 84, 15589, 2962, 7, 157, 845, 5, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [9310, 1550, 21, 11301, 6, 4198, 591, 48, 1379, 6, 1315, 12, 15589, 2962, 7, 157, 5, 1]}


In [ ]:
from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

# Prepare data collator
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

# Training configuration
training_args = Seq2SeqTrainingArguments(
    output_dir="./gec_t5",
    per_device_train_batch_size=32,
    num_train_epochs=1,
    learning_rate=3e-4,
    logging_steps=100,
    save_strategy="epoch",
    fp16=False,
    report_to="none"
)

# Create trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Training configuration completed!")

Training configuration completed!


In [ ]:
trainer.train()

Step,Training Loss
100,0.959477


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=157, training_loss=0.9281139252292123, metrics={'train_runtime': 47.3672, 'train_samples_per_second': 105.558, 'train_steps_per_second': 3.315, 'total_flos': 84504037294080.0, 'train_loss': 0.9281139252292123, 'epoch': 1.0})

In [ ]:
# Save model and tokenizer
model.save_pretrained("./gec_t5")
tokenizer.save_pretrained("./gec_t5")

print("Fine-tuned model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved successfully!


In [ ]:
def correct_grammar(sentence):
    input_text = "grammar: " + sentence

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        max_length=64,
        truncation=True
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_length=64,
        num_beams=4,
        early_stopping=True
    )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )


test_sentences = [
    "She go to school yesterday and buyed a book.",
    "He don't like apples.",
    "I has a important meeting at tomorrow.",
    "Their going to the market for buy vegetables."
]

for sentence in test_sentences:
    print("\nOriginal :", sentence)
    print("Corrected:", correct_grammar(sentence))


Original : She go to school yesterday and buyed a book.
Corrected: She go to school yesterday and bought a book.

Original : He don't like apples.
Corrected: He doesn't like apples.

Original : I has a important meeting at tomorrow.
Corrected: I have a important meeting at tomorrow.

Original : Their going to the market for buy vegetables.
Corrected: Their going to the market for buy vegetables.


In [ ]:
from datasets import load_dataset

# Load JFLEG validation dataset
jfleg = load_dataset(
    "jhu-clsp/jfleg",
    split="validation"
)

print("JFLEG validation examples:", len(jfleg))
print("\nSample:")
print(jfleg[0])

README.md:   0%|          | 0.00/5.94k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  148kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  141kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/755 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/748 [00:00<?, ? examples/s]

JFLEG validation examples: 755

Sample:
{'sentence': 'So I think we can not live if old people could not find siences and tecnologies and they did not developped . ', 'corrections': ['So I think we would not be alive if our ancestors did not develop sciences and technologies . ', 'So I think we could not live if older people did not develop science and technologies . ', 'So I think we can not live if old people could not find science and technologies and they did not develop . ', 'So I think we can not live if old people can not find the science and technology that has not been developed . ']}


In [ ]:
import nltk
from nltk.translate.gleu_score import corpus_gleu

# Download tokenizer resources if needed
nltk.download("punkt", quiet=True)

model.eval()

hypotheses = []
references = []

for item in jfleg:

    sentence = item["sentence"]

    # Generate model correction
    predicted = correct_grammar(sentence)

    # Tokenize prediction
    hypotheses.append(predicted.split())

    # Tokenize all human references
    refs = [
        correction.split()
        for correction in item["corrections"]
    ]

    references.append(refs)

# Calculate GLEU score
gleu_score = corpus_gleu(
    references,
    hypotheses
)

print("JFLEG validation examples:", len(hypotheses))
print("Fine-tuned model GLEU:", round(gleu_score, 4))

JFLEG validation examples: 755
Fine-tuned model GLEU: 0.601


In [ ]:
# Move fine-tuned model to CPU to free GPU memory
model = model.to("cpu")

# Load original, pre-trained T5-small
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(
    "t5-small"
).to("cuda")

baseline_model.eval()

baseline_hypotheses = []
baseline_references = []

for item in jfleg:

    sentence = "grammar: " + item["sentence"]

    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        max_length=64,
        truncation=True
    ).to("cuda")

    outputs = baseline_model.generate(
        **inputs,
        max_length=64,
        num_beams=4,
        early_stopping=True
    )

    predicted = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    baseline_hypotheses.append(predicted.split())

    refs = [
        correction.split()
        for correction in item["corrections"]
    ]

    baseline_references.append(refs)


baseline_gleu = corpus_gleu(
    baseline_references,
    baseline_hypotheses
)

print("Original T5-small GLEU:", round(baseline_gleu, 4))
print("Fine-tuned T5-small GLEU:", 0.601)

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

Original T5-small GLEU: 0.5092
Fine-tuned T5-small GLEU: 0.601


In [ ]:
import shutil

# Create a compressed model archive
shutil.make_archive(
    "/content/gec_t5",
    "zip",
    "/content/gec_t5"
)

print("Model compressed successfully!")

Model compressed successfully!


In [ ]:
from google.colab import files

files.download("/content/gec_t5.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

print("Current directory:", os.getcwd())
print("\nFiles in /content:")

for item in os.listdir("/content"):
    print(item)

print("\nDataset variable available:", "ds" in globals())
print("Original model directory exists:", os.path.exists("/content/gec_t5"))

Current directory: /content

Files in /content:
.config
sample_data

Dataset variable available: False
Original model directory exists: False


In [ ]:
!pip install -q transformers datasets sentencepiece accelerate nltk

In [ ]:
from datasets import load_dataset
import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

ds = load_dataset(
    "martinsr/c4_200m",
    split="train",
    streaming=True
)

print("Dataset loaded successfully!")

CUDA available: True
GPU: Tesla T4


README.md:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/62 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/62 [00:00<?, ?it/s]

Dataset loaded successfully!


In [ ]:
improved_pairs = [
    (item["input"], item["output"])
    for item in ds.take(20000)
]

improved_pairs = [
    (incorrect.strip(), correct.strip())
    for incorrect, correct in improved_pairs
    if isinstance(incorrect, str)
    and isinstance(correct, str)
    and incorrect.strip()
    and correct.strip()
]

print("Training examples:", len(improved_pairs))
print("Sample:", improved_pairs[0])

Training examples: 20000
Sample: ('Bitcoin is for $7,094 this morning, which CoinDesk says.', 'Bitcoin goes for $7,094 this morning, according to CoinDesk.')


In [ ]:
from datasets import Dataset

# Convert sentence pairs into a Dataset
full_dataset = Dataset.from_dict({
    "input": [item[0] for item in improved_pairs],
    "output": [item[1] for item in improved_pairs]
})

# Split into training and validation sets
split_dataset = full_dataset.train_test_split(
    test_size=0.05,
    seed=42
)

train_dataset = split_dataset["train"]
validation_dataset = split_dataset["test"]

print("Training examples:", len(train_dataset))
print("Validation examples:", len(validation_dataset))

Training examples: 19000
Validation examples: 1000


In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

model = model.to("cuda")

print("Fresh T5-small loaded!")
print("Device:", next(model.parameters()).device)

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Fresh T5-small loaded!
Device: cuda:0


In [ ]:
def preprocess_function(examples):

    inputs = [
        "grammar: " + sentence
        for sentence in examples["input"]
    ]

    model_inputs = tokenizer(
        inputs,
        max_length=64,
        truncation=True
    )

    labels = tokenizer(
        text_target=examples["output"],
        max_length=64,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


# Tokenize training data
tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True
)

# Tokenize validation data
tokenized_validation = validation_dataset.map(
    preprocess_function,
    batched=True
)

print("Tokenization completed!")
print("Training examples:", len(tokenized_train))
print("Validation examples:", len(tokenized_validation))

Map:   0%|          | 0/19000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenization completed!
Training examples: 19000
Validation examples: 1000


In [ ]:
from transformers import (
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./gec_t5_improved",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    learning_rate=1e-4,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    fp16=False,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_validation,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Improved training configuration ready!")

Improved training configuration ready!


In [ ]:
training_results = trainer.train()

print("\nImproved training completed!")
print("Final training loss:", training_results.training_loss)

Epoch,Training Loss,Validation Loss
1,0.795646,0.785619
2,0.746769,0.784628
3,0.758480,0.783684


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Improved training completed!
Final training loss: 0.7681898316148957


In [ ]:
# Save improved model and tokenizer
trainer.save_model("./gec_t5_improved")
tokenizer.save_pretrained("./gec_t5_improved")

print("Improved model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Improved model saved successfully!


In [ ]:
from datasets import load_dataset

jfleg = load_dataset(
    "jhu-clsp/jfleg",
    split="validation"
)

print("JFLEG examples:", len(jfleg))

README.md:   0%|          | 0.00/5.94k [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  148kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  141kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/755 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/748 [00:00<?, ? examples/s]

JFLEG examples: 755


In [ ]:
from nltk.translate.gleu_score import corpus_gleu

model.eval()

hypotheses = []
references = []

for item in jfleg:

    sentence = "grammar: " + item["sentence"]

    inputs = tokenizer(
        sentence,
        return_tensors="pt",
        max_length=64,
        truncation=True
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_length=64,
        num_beams=4,
        early_stopping=True
    )

    predicted = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    hypotheses.append(predicted.split())

    refs = [
        correction.split()
        for correction in item["corrections"]
    ]

    references.append(refs)

improved_gleu = corpus_gleu(
    references,
    hypotheses
)

print("Improved model GLEU:", round(improved_gleu, 4))
print("Previous model GLEU: 0.6010")

Improved model GLEU: 0.6057
Previous model GLEU: 0.6010


In [ ]:
test_sentences = [
    "She go to school yesterday and buyed a book.",
    "He don't like apples.",
    "I has a important meeting at tomorrow.",
    "Their going to the market for buy vegetables."
]

model.eval()

for sentence in test_sentences:
    prompt = "grammar: " + sentence

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=64,
        truncation=True
    ).to("cuda")

    outputs = model.generate(
        **inputs,
        max_length=64,
        num_beams=4,
        early_stopping=True
    )

    corrected = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    print("Original:  ", sentence)
    print("Corrected: ", corrected)
    print("-" * 60)

Original:   She go to school yesterday and buyed a book.
Corrected:  She went to school yesterday and bought a book.
------------------------------------------------------------
Original:   He don't like apples.
Corrected:  He doesn't like apples.
------------------------------------------------------------
Original:   I has a important meeting at tomorrow.
Corrected:  I have a important meeting at tomorrow.
------------------------------------------------------------
Original:   Their going to the market for buy vegetables.
Corrected:  Their going to the market for vegetables.
------------------------------------------------------------


In [ ]:
import shutil

shutil.make_archive(
    "/content/gec_t5_improved",
    "zip",
    "/content/gec_t5_improved"
)

from google.colab import files
files.download("/content/gec_t5_improved.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>